# Analyse de données bancaires avec Python

**Profils de clients associés au statut observé Actif / Inactif**

## 1. Contexte du projet

Une banque dispose d'un jeu de données contenant des informations sur 10 000 clients :
âge, ville de résidence, revenu annuel, type de compte, solde bancaire et statut (Actif ou Inactif).

Elle souhaite mieux comprendre les caractéristiques des clients inactifs et identifier
les différences entre clients actifs et inactifs.

## 2. Problématique

Quelles caractéristiques démographiques et financières sont associées au statut d'activité des clients ?

## 3. Objectifs

- Décrire la population étudiée.
- Analyser les variables démographiques et financières.
- Comparer les clients actifs et inactifs.
- Tester statistiquement les associations entre les variables et le statut.
- Évaluer si une classification des clients selon leur statut est possible.

## 4. Limites de l'étude

Le jeu de données est une photographie des clients à un instant donné. Il ne contient aucun
historique : il ne permet donc ni d'étudier l'évolution du statut, ni de prédire une inactivité future.
Les résultats décrivent des **associations**, jamais des relations de causalité.

## 5. Organisation du notebook

Ce notebook **lit** les couches produites par le pipeline (`python -m src.pipeline`) et réutilise
les fonctions de `src/`. Il n'écrit aucun fichier de données : la logique de transformation vit
dans les scripts, le notebook sert à explorer et à interpréter.

| Partie | Contenu |
|---|---|
| A | Préparation de l'environnement |
| B | Couche Bronze : audit de la source |
| C | Couche Silver : nettoyage et décisions de qualité |
| D | Analyse exploratoire |
| E | Couche Gold Analytics : tables de reporting |
| F | Tests d'association |
| G | Classification : comparaison de modèles |
| H | Conclusion |

## 6. Questions analytiques

**Description générale** — Combien de clients ? Quelle répartition Actif / Inactif ? Quels types de comptes ?

**Démographie** — Quel est l'âge moyen ? Les profils d'âge diffèrent-ils selon le statut ?

**Finances** — Quels sont le revenu et le solde moyens ? Diffèrent-ils entre actifs et inactifs ?

**Comptes** — La proportion de clients inactifs varie-t-elle selon le type de compte ?

**Machine Learning** — Quelles variables sont associées au statut ? Un modèle peut-il distinguer actifs et inactifs ?

## A. Préparation de l'environnement

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Racine du projet : premier dossier parent contenant src/config.py
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.utils.io import read_source_csv

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
PALETTE = {"Actif": "#2E8B57", "Inactif": "#C0392B"}


def relative(path: Path) -> str:
    """Chemin relatif à la racine du projet (jamais de chemin absolu dans les sorties)."""
    return Path(path).relative_to(PROJECT_ROOT).as_posix()


required = [config.BRONZE_FILE, config.SILVER_FILE, config.GOLD_ML_FILE,
            config.GOLD_ANALYTICS_DIR / "gold_fact_clients.parquet"]
missing = [relative(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Fichiers absents : {missing}. Exécutez d'abord `python -m src.pipeline` à la racine du projet."
    )
print("Couches disponibles :", [relative(p) for p in required])

## B. Couche Bronze — audit de la source

La couche Bronze est une copie fidèle de la source. Elle contient des identifiants directs
(`nom`, `prenom`).

**Règle de confidentialité du notebook :** ces deux colonnes sont retirées dès le chargement de
l'aperçu. Aucune cellule n'affiche de nom ni de prénom, car les sorties d'un notebook sont
publiées avec lui.

In [ ]:
df_bronze = read_source_csv(config.BRONZE_FILE)

# La source contient des identifiants et des données financières individuelles.
# On affiche uniquement le schéma général, jamais des lignes de clients.
print("Dimensions de la source :", df_bronze.shape)
print("Nombre de colonnes :", len(df_bronze.columns))
print("Enregistrements individuels masqués dans cette présentation pour protéger les clients.")

In [ ]:
audit = pd.DataFrame({
    "type": df_bronze.dtypes.astype(str),
    "valeurs_manquantes": df_bronze.isna().sum(),
    "pourcentage_manquant": (df_bronze.isna().mean() * 100).round(2),
    "valeurs_distinctes": df_bronze.nunique(),
})
audit

In [ ]:
doublons_exacts = int(df_bronze.duplicated().sum())
sans_doublons = df_bronze.drop_duplicates()

print("Lignes dupliquées exactement :", doublons_exacts)
print("Lignes après dédoublonnage :", len(sans_doublons))
print("client_id distincts :", df_bronze["client_id"].nunique())
print("client_id encore dupliqués après dédoublonnage :", int(sans_doublons["client_id"].duplicated().sum()))

### B.1 — Constats de l'audit

- La source contient 10 colonnes et un identifiant technique par client.
- Les doublons exacts sont supprimés ; les identifiants techniques restants doivent être uniques.
- La colonne `age` comporte des valeurs manquantes ; elles sont documentées et conservées à ce stade.
- Les types lus sont confrontés au dictionnaire de données (`docs/data_dictionary.md`).

Le rapport détaillé est généré localement par `python -m src.audit.audit_source` dans `reports/audit/`.


## C. Couche Silver — nettoyage et décisions de qualité

La couche Silver est construite par `src/silver/build_silver.py`. Les règles appliquées sont
documentées dans `docs/data_quality_rules.md` :

1. suppression des doublons exacts ;
2. unicité de `client_id` ;
3. types explicites (âge entier *nullable*) ;
4. catégories autorisées pour `statut` et `type_compte` ;
5. bornes de cohérence (âge entre 18 et 120 ans, revenu et solde non négatifs, population positive) ;
6. une seule population par ville ;
7. retrait de `nom` et `prenom` (pseudonymisation).

Si une règle n'est pas respectée, le script s'arrête avec une erreur : un fichier Silver présent
sur le disque est donc un fichier validé.

In [ ]:
df_silver = pd.read_parquet(config.SILVER_FILE)

print("Dimensions Bronze :", df_bronze.shape)
print("Dimensions Silver :", df_silver.shape)
print("Identifiants directs présents en Silver :",
      sorted(set(config.DIRECT_IDENTIFIERS) & set(df_silver.columns)) or "aucun")
df_silver.dtypes.astype(str).to_frame("type")

### C.1 — Traitement des âges manquants

Avant toute imputation, on examine la fréquence des valeurs manquantes selon le statut et leur distribution.
La décision de ce projet est de conserver ces valeurs manquantes dans Silver afin de ne pas fabriquer de valeurs observées.


In [ ]:
age_par_statut = df_silver.groupby("statut")["age"].agg(
    clients="size",
    ages_renseignes="count",
    ages_manquants=lambda s: int(s.isna().sum()),
    moyenne="mean",
    mediane="median",
    ecart_type="std",
)
age_par_statut

**Décision : les âges manquants sont conservés tels quels dans la couche Silver.**

- L'absence d'une valeur n'est pas remplacée par une moyenne arbitraire dans la couche de stockage.
- Une imputation avant la séparation entraînement / test ferait intervenir des informations du jeu de test dans le prétraitement (fuite de données).
- Les analyses descriptives indiquent explicitement le nombre d'âges renseignés et manquants.

En conséquence :
- dans Gold Analytics, les clients dont l'âge manque forment une tranche « Non renseigné » ;
- pour le Machine Learning, l'imputation (médiane) est apprise dans le pipeline scikit-learn sur les seules données d'entraînement.


### C.2 — Valeurs atypiques (méthode IQR)

Une valeur est dite atypique si elle se situe sous `Q1 − 1,5 × IQR` ou au-dessus de
`Q3 + 1,5 × IQR`. Atypique ne veut pas dire erronée : la décision de traitement dépend du contexte métier.

In [ ]:
numeric_cols = ["age", "population_ville", "revenu_annuel", "solde"]
lignes = []
for col in numeric_cols:
    valeurs = df_silver[col].dropna().astype(float)
    q1, q3 = valeurs.quantile([0.25, 0.75])
    iqr = q3 - q1
    borne_inf, borne_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_atypiques = int(((valeurs < borne_inf) | (valeurs > borne_sup)).sum())
    lignes.append({"variable": col, "Q1": q1, "Q3": q3, "borne_inferieure": borne_inf,
                   "borne_superieure": borne_sup, "valeurs_atypiques": n_atypiques,
                   "pourcentage": n_atypiques / len(df_silver) * 100})
outliers = pd.DataFrame(lignes)
outliers

In [ ]:
borne_sup_population = outliers.set_index("variable").loc["population_ville", "borne_superieure"]
(
    df_silver[df_silver["population_ville"] > borne_sup_population]
    .groupby("ville")
    .agg(population_ville=("population_ville", "first"), clients=("client_id", "size"))
    .sort_values("population_ville", ascending=False)
)

**Interprétation.** Les seuils IQR identifient les valeurs atypiques au sens statistique, mais une valeur atypique n'est pas automatiquement une erreur.
Les populations de grandes villes doivent être vérifiées à partir d'une source de référence datée avant toute correction.
**Décision : aucune observation n'est supprimée sur la seule base du test IQR.**


## D. Analyse exploratoire

### D.1 — Variables numériques

In [ ]:
df_silver[numeric_cols].astype(float).describe().T

- **Âge** : les statistiques sont calculées uniquement sur les valeurs renseignées ; le nombre de valeurs manquantes est signalé séparément.
- **Population de la ville** : la moyenne peut être influencée par les grandes villes ; la médiane et la source de référence sont donc utiles.
- **Revenu annuel** et **solde** : les unités et périodes de référence doivent être confirmées dans la documentation métier.


### D.2 — Variables catégorielles

In [ ]:
for col in ["statut", "type_compte"]:
    repartition = df_silver[col].value_counts().to_frame("effectif")
    repartition["pourcentage"] = (repartition["effectif"] / len(df_silver) * 100).round(2)
    display(repartition)

villes = df_silver["ville"].value_counts()
print(f"Villes : {villes.size} | effectif minimal : {villes.min()} | effectif maximal : {villes.max()}")

La répartition des statuts, des types de comptes et des villes décrit la composition du portefeuille. Ces répartitions constituent un point de départ descriptif ; elles ne suffisent pas à expliquer le statut d'un client.


### D.3 — Statut selon le type de compte

In [ ]:
statut_par_compte = pd.crosstab(df_silver["type_compte"], df_silver["statut"])
statut_par_compte_pct = pd.crosstab(df_silver["type_compte"], df_silver["statut"], normalize="index") * 100
display(statut_par_compte)
display(statut_par_compte_pct.round(2))

ax = statut_par_compte_pct.plot(kind="bar", stacked=True, figsize=(8, 4.5),
                                color=[PALETTE["Actif"], PALETTE["Inactif"]])
ax.set(title="Répartition des statuts par type de compte", xlabel="Type de compte", ylabel="Pourcentage (%)", ylim=(0, 100))
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Statut", loc="lower right")
plt.tight_layout()
plt.show()

La comparaison des taux par type de compte est descriptive. La partie F applique un test global d'association avant toute interprétation métier ou recommandation de ciblage.


### D.4 — Âge, revenu et solde selon le statut

In [ ]:
resume = (
    df_silver.assign(age=df_silver["age"].astype(float))
    .groupby("statut")[["age", "revenu_annuel", "solde"]]
    .agg(["mean", "median", "std"])
)
resume

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
donnees = df_silver.assign(age=df_silver["age"].astype(float))
for ax, (col, titre) in zip(axes, [("age", "Âge (années)"), ("revenu_annuel", "Revenu annuel"), ("solde", "Solde bancaire")]):
    sns.boxplot(data=donnees, x="statut", y=col, hue="statut", palette=PALETTE, legend=False, ax=ax)
    ax.set(title=titre, xlabel="Statut", ylabel="")
fig.suptitle("Distributions selon le statut du client")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (col, titre) in zip(axes, [("age", "Âge (années)"), ("revenu_annuel", "Revenu annuel"), ("solde", "Solde bancaire")]):
    sns.histplot(data=donnees, x=col, hue="statut", palette=PALETTE, bins=15,
                 element="step", stat="count", common_norm=False, ax=ax)
    ax.set(title=titre, xlabel="", ylabel="Nombre de clients")
fig.suptitle("Histogrammes superposés selon le statut")
plt.tight_layout()
plt.show()

Les distributions et statistiques par statut doivent être comparées à la fois par leur niveau central et leur dispersion. Un écart de moyenne isolé ne suffit pas à établir une différence réelle ; la partie F apporte les tests d'association et les tailles d'effet.


### D.5 — Corrélations entre variables numériques

In [ ]:
correlations = df_silver[numeric_cols].astype(float).corr()
plt.figure(figsize=(6.5, 5))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matrice de corrélation (Pearson)")
plt.tight_layout()
plt.show()

hors_diagonale = correlations.where(~np.eye(len(correlations), dtype=bool)).abs().max().max()
print(f"Plus forte corrélation hors diagonale (valeur absolue) : {hors_diagonale:.3f}")

La matrice de corrélation permet de repérer des relations **linéaires** entre variables quantitatives. Des coefficients faibles indiquent qu'aucune relation linéaire marquée n'est visible dans cet échantillon ; ils ne prouvent ni l'indépendance statistique générale ni l'origine synthétique des données.


## E. Couche Gold Analytics — tables de reporting

Les tables Gold sont construites par `src/gold/build_gold_analytics.py` et alimenteront le rapport
Power BI. Chaque table est réconciliée avec Silver (mêmes totaux) avant d'être écrite.

In [ ]:
gold = {
    path.stem: pd.read_parquet(path)
    for path in sorted(config.GOLD_ANALYTICS_DIR.glob("gold_*.parquet"))
}
pd.DataFrame(
    [{"table": nom, "lignes": t.shape[0], "colonnes": t.shape[1]} for nom, t in gold.items()]
)

In [ ]:
gold["gold_kpi_global"].T.rename(columns={0: "valeur"})

In [ ]:
display(gold["gold_age_analysis"])
display(gold["gold_income_analysis"])
display(gold["gold_balance_analysis"])

Les tableaux Gold décrivent les segments selon l'âge, le revenu et le solde. Les différences observées entre segments doivent être interprétées à la lumière des tests statistiques et ne constituent pas, à elles seules, des leviers d'action.

### E.1 — Analyse par ville : écarts réels ou bruit ?

Les taux par ville sont des estimations calculées sur des effectifs finis. Les intervalles de confiance aident à visualiser leur incertitude ; un classement brut peut exagérer des écarts dus à l'échantillonnage.


In [ ]:
villes = gold["gold_city_analysis"].copy()
p = villes["taux_inactivite_pct"] / 100
n = villes["total_clients"]
z = 1.96
# Intervalle de confiance de Wilson à 95 %
centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
demi_largeur = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
villes["ic_bas"], villes["ic_haut"] = (centre - demi_largeur) * 100, (centre + demi_largeur) * 100
villes = villes.sort_values("taux_inactivite_pct")

taux_global = gold["gold_kpi_global"].loc[0, "taux_inactivite_pct"]
fig, ax = plt.subplots(figsize=(8, 9))
ax.errorbar(villes["taux_inactivite_pct"], villes["ville"],
            xerr=[villes["taux_inactivite_pct"] - villes["ic_bas"], villes["ic_haut"] - villes["taux_inactivite_pct"]],
            fmt="o", color="#34495E", ecolor="#95A5A6", capsize=3)
ax.axvline(taux_global, color=PALETTE["Inactif"], linestyle="--", label=f"Taux global ({taux_global:.2f} %)")
ax.set(title="Taux d'inactivité par ville (intervalle de confiance à 95 %)", xlabel="Taux d'inactivité (%)")
ax.legend()
plt.tight_layout()
plt.show()

hors_ic = villes[(villes["ic_bas"] > taux_global) | (villes["ic_haut"] < taux_global)]
print(f"Villes dont l'intervalle exclut le taux global : {len(hors_ic)} sur {len(villes)}"
      + (f" ({', '.join(hors_ic['ville'])})" if len(hors_ic) else ""))

Les intervalles de confiance sont un outil descriptif, pas un classement commercial. Les villes ne doivent pas être désignées comme « meilleures » ou « moins bonnes » sur la seule base de leurs taux observés. Le test global de la partie F complète cette lecture.


## F. Tests d'association

Les graphiques suggèrent des différences parfois faibles ; les tests statistiques évaluent si les données apportent une preuve suffisante d'association :

- variables catégorielles : test du khi-deux et V de Cramér ;
- variables numériques : test de Mann-Whitney et AUC ;
- p-valeurs corrigées pour comparaisons multiples (méthode de Holm).

L'absence de significativité ne démontre pas une indépendance parfaite : elle signifie que l'étude ne détecte pas d'association au seuil retenu.


In [ ]:
from src.analysis.association_tests import run_association_tests

tests = run_association_tests(gold["gold_fact_clients"])
with pd.option_context("display.float_format", lambda v: f"{v:.4f}"):
    display(tests[["variable", "test", "p_valeur", "p_valeur_holm", "taille_effet", "mesure_effet", "significatif_5pct"]])
print("Associations significatives au seuil de 5 % :", int(tests["significatif_5pct"].sum()))

Les tests ne mettent pas en évidence d'association statistiquement significative au seuil de 5 % après correction de Holm. Les tailles d'effet observées sont faibles dans cet échantillon.

Cela ne prouve pas que le statut soit strictement indépendant de ces variables : cela indique qu'aucune association suffisamment étayée n'a été détectée avec les variables disponibles et les tests retenus. Les écarts descriptifs ne doivent donc pas être convertis en recommandations commerciales.


## G. Classification — comparaison de modèles

**Objectif :** vérifier si une combinaison de variables, éventuellement non linéaire, permet de
distinguer actifs et inactifs là où chaque variable prise isolément n'y parvient pas.

**Protocole** (`src/ml/train_models.py`) :

1. séparation stratifiée entraînement / test (80 % / 20 %, graine fixe) ;
2. prétraitement **dans** le pipeline : imputation médiane de l'âge, standardisation, encodage one-hot —
   appris sur les seules données d'entraînement ;
3. une baseline (`DummyClassifier`, classe majoritaire) et quatre algorithmes ;
4. validation croisée à 5 plis sur le jeu d'entraînement pour comparer les modèles ;
5. évaluation unique sur le jeu de test ;
6. test de permutation : le score obtenu est-il meilleur que ceux obtenus en mélangeant la cible ?

La classe d'intérêt est `Inactif`. La cible n'est jamais incluse dans les variables explicatives.

In [ ]:
from src.ml.train_models import BASELINE_NAME, evaluate_models, load_xy

gold_ml = pd.read_parquet(config.GOLD_ML_FILE)
X, y = load_xy(gold_ml)

print("Variables explicatives :", X.columns.tolist())
print("Âges manquants (imputés dans le pipeline) :", int(X["age"].isna().sum()))
print("Part de clients inactifs :", round(float(y.mean()) * 100, 2), "%")

In [ ]:
resultats = evaluate_models(X, y, n_permutations=200)
comparaison = resultats["comparison"]
with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
    display(comparaison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

donnees_cv = comparaison.sort_values("cv_roc_auc_moyenne")
axes[0].barh(donnees_cv["modele"], donnees_cv["cv_roc_auc_moyenne"],
             xerr=donnees_cv["cv_roc_auc_ecart_type"], color="#4C72B0", capsize=4)
axes[0].axvline(0.5, color=PALETTE["Inactif"], linestyle="--", label="Hasard (0,5)")
axes[0].set(xlim=(0.40, 0.65), xlabel="ROC-AUC (moyenne ± écart-type)", title="Validation croisée à 5 plis")
axes[0].legend()

permutation = resultats["permutation_test"]
axes[1].axvline(permutation["roc_auc_reel"], color="#4C72B0", linewidth=2, label="Score réel")
axes[1].axvline(permutation["roc_auc_permutations_p95"], color="grey", linestyle=":", label="95e centile des permutations")
axes[1].axvline(0.5, color=PALETTE["Inactif"], linestyle="--", label="Hasard (0,5)")
axes[1].set(xlim=(0.45, 0.55), yticks=[], xlabel="ROC-AUC", title="Test de permutation (régression logistique)")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Score réel : {permutation['roc_auc_reel']:.3f} | "
      f"95e centile avec cible mélangée : {permutation['roc_auc_permutations_p95']:.3f} | "
      f"p-valeur : {permutation['p_valeur']:.3f}")

In [ ]:
modeles = [m for m in resultats["confusion_matrices"] if m != BASELINE_NAME]
fig, axes = plt.subplots(1, len(modeles), figsize=(4 * len(modeles), 3.8))
for ax, nom in zip(axes, modeles):
    sns.heatmap(resultats["confusion_matrices"][nom], annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Actif", "Inactif"], yticklabels=["Actif", "Inactif"], ax=ax)
    ax.set(title=nom, xlabel="Statut prédit", ylabel="Statut réel")
fig.suptitle("Matrices de confusion — jeu de test (2 000 clients)")
plt.tight_layout()
plt.show()

### G.1 — Lecture des résultats

- Les ROC-AUC de validation croisée sont proches de 0,5, niveau correspondant à une capacité de classement proche du hasard.
- Le test de permutation ne fournit pas de preuve que la régression logistique exploite un signal supérieur à celui obtenu avec une cible mélangée.
- L'accuracy doit être comparée à la baseline ; une valeur légèrement supérieure sur un seul jeu de test ne suffit pas à sélectionner un modèle.
- Les importances issues d'arbres ne sont pas interprétées lorsque le pouvoir discriminant n'est pas établi : un modèle peut attribuer des importances à du bruit.

**Conclusion de modélisation : aucun modèle n'est retenu pour un usage opérationnel sur la base de cette expérience.**

### G.2 — Pourquoi ne pas interpréter l'importance des variables ?

Un arbre ou une forêt aléatoire peut attribuer des importances même lorsqu'il apprend principalement du bruit. Sans pouvoir discriminant validé, ces importances ne constituent pas une preuve d'influence sur le statut.


## H. Conclusion

### Réponses aux questions analytiques

| Question | Réponse méthodologique |
|---|---|
| Combien de clients ? | Le total est calculé après retrait des doublons exacts et contrôle de l'unicité de `client_id`. |
| Répartition des statuts ? | Les effectifs et proportions Actif / Inactif sont calculés dans Gold Analytics. |
| Types de comptes ? | La composition du portefeuille est décrite par type de compte. |
| Âge moyen ? | La moyenne est calculée sur les âges renseignés ; les valeurs manquantes restent identifiées. |
| Revenu et solde moyens ? | Les moyennes, médianes et distributions sont calculées et exportées. |
| Différences entre actifs et inactifs ? | Les comparaisons descriptives sont complétées par des tests d'association et leurs tailles d'effet. |
| Un modèle peut-il distinguer les statuts ? | La décision repose sur la validation croisée, la baseline, le jeu de test et le test de permutation. |

### Enseignement principal

Cette étude porte sur le **statut observé** à la date d'extraction. Les tests d'association et la classification servent à vérifier si les variables disponibles apportent un signal exploitable ; ils ne démontrent jamais une causalité.

Lorsque les variables ne permettent pas de distinguer les statuts de façon convaincante, la conclusion opérationnelle est de ne pas déployer un score de ciblage et de ne pas classer les villes ou les catégories de clients comme leviers d'action.

### Ce que l'étude ne permet pas de dire

- Rien sur l'inactivité **future** : il n'y a pas d'historique temporel.
- Rien sur les **causes** de l'inactivité.
- Aucun ciblage commercial par ville, type de compte ou tranche d'âge sans preuve supplémentaire.

### Recommandations et suite du projet

Les recommandations métier sont détaillées dans `docs/recommendations.md`. Le rapport Power BI décrit le portefeuille sans présenter des écarts descriptifs comme des facteurs causaux (`powerbi/README.md`).
